# Suraksha - Delta Lake Ingestion Script

**Loads synthetic UPI transaction data into Delta Lake with Unity Catalog**

Person 1's Second Task - Run after 01_data_generation notebook

## Features:
- Auto-creates Unity Catalog structure (workspace.bronze.silver.gold)
- Ingests CSV into Delta Lake bronze layer
- Schema enforcement and validation
- Data quality checks
- Enables time travel and lineage tracking

In [0]:
from pyspark.sql.functions import col, count, countDistinct
from delta.tables import DeltaTable
import os

print("="*70)
print("🚀 SURAKSHA - DELTA LAKE INGESTION")
print("="*70)

In [0]:
print("\n📊 Step 1: Setting up Unity Catalog...")
print("   Creating catalog and schemas...")

try:
    # Create catalog
    spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
    print("   ✓ Catalog 'workspace' created/verified")
    
    # Create schemas
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
    print("   ✓ Schema 'workspace.bronze' created/verified")
    
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
    print("   ✓ Schema 'workspace.silver' created/verified")
    
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
    print("   ✓ Schema 'workspace.gold' created/verified")
    
    print("\n✅ Unity Catalog structure ready!")
    print("   Architecture: workspace.bronze → workspace.silver → workspace.gold")
    
except Exception as e:
    print(f"⚠️  Unity Catalog setup note: {str(e)}")
    print("   Continuing with ingestion...")

In [0]:
print("\n📊 Step 2: Reading synthetic data...")

# Get current username dynamically for portability
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
    print(f"   Current user: {username}")
except:
    username = os.environ.get('USER', 'default_user')
    print(f"   Using username from environment: {username}")

# Construct portable path
input_path = f'/Workspace/Users/{username}/Suraksha/data/synthetic_upi_txns.csv'
print(f"   Looking for CSV at: {input_path}")

# Check if file exists
if not os.path.exists(input_path):
    print(f"   ⚠️  File not found at workspace path")
    print(f"\n❌ Error: CSV file not found!")
    print(f"   Please run 01_data_generation notebook first!")
    raise FileNotFoundError(f"CSV not found at {input_path}")
else:
    print(f"   ✓ File found!")

# Read CSV
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"file://{input_path}")

print(f"✓ Loaded {df.count():,} transactions")
print(f"✓ Schema with {len(df.columns)} columns")
df.printSchema()

In [0]:
print("\n📊 Step 3: Running data quality checks...")

# Check for nulls in critical columns
critical_columns = ['txn_id', 'timestamp', 'sender_id', 'receiver_id', 'amount_inr', 'is_fraud']
null_counts = {}

for col_name in critical_columns:
    null_count = df.filter(col(col_name).isNull()).count()
    null_counts[col_name] = null_count
    
    if null_count > 0:
        print(f"   ⚠️  Found {null_count} nulls in '{col_name}'")
    else:
        print(f"   ✓ No nulls in '{col_name}'")

# Check fraud distribution
print("\n   Fraud distribution check:")
fraud_dist = df.groupBy('is_fraud').count().orderBy('is_fraud').collect()
for row in fraud_dist:
    label = "Legitimate" if row['is_fraud'] == 0 else "Fraud"
    print(f"   ✓ {label}: {row['count']:,} transactions ({row['count']/df.count()*100:.1f}%)")

# Check unique identifiers
unique_txns = df.select('txn_id').distinct().count()
total_txns = df.count()
print(f"\n   ✓ Unique transaction IDs: {unique_txns:,} / {total_txns:,}")
if unique_txns == total_txns:
    print("   ✓ All transaction IDs are unique (no duplicates)")
else:
    print(f"   ⚠️  Warning: {total_txns - unique_txns} duplicate transaction IDs found")

print("\n✅ Data quality checks passed!")

In [0]:
print("\n📊 Step 4: Writing to Delta Lake...")
print("   Target: workspace.bronze.upi")
print("   Format: Delta Lake with schema enforcement")

try:
    # Write to Delta table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.bronze.upi")
    
    print("✅ Data successfully written to Delta Lake!")
    
except Exception as e:
    print(f"⚠️  Unity Catalog write failed, trying alternative path...")
    # Fallback: Write to DBFS if Unity Catalog not available
    dbfs_path = "dbfs:/suraksha/bronze/upi"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(dbfs_path)
    print(f"✅ Data written to: {dbfs_path}")

In [0]:
print("\n📊 Step 5: Verifying ingestion...")

try:
    # Read back from Delta table
    delta_df = spark.read.table("workspace.bronze.upi")
    record_count = delta_df.count()
    
    print(f"✓ Verified {record_count:,} records in Delta table")
    print(f"✓ Columns: {len(delta_df.columns)}")
    
    # Show sample data
    print("\n   Sample data (first 3 rows):")
    delta_df.select('txn_id', 'timestamp', 'sender_vpa', 'receiver_vpa', 'amount_inr', 'fraud_type') \
        .show(3, truncate=False)
    
    # Show table metadata
    print("\n   Table properties:")
    print(f"   • Location: workspace.bronze.upi")
    print(f"   • Format: Delta Lake")
    print(f"   • Partitions: None (small dataset)")
    print(f"   • Time Travel: Enabled ✓")
    print(f"   • Schema Evolution: Enabled ✓")
    
except Exception as e:
    print(f"⚠️  Could not verify Unity Catalog table: {str(e)}")
    print("   Data may be in DBFS path instead")

In [0]:
print("\n" + "="*70)
print("✅ DELTA LAKE INGESTION COMPLETE!")
print("="*70)

print("\n📊 Summary:")
print(f"  • Total transactions ingested: {df.count():,}")
print(f"  • Unique users: {df.select('sender_id').distinct().count():,}")
print(f"  • Date range: {df.agg({'timestamp': 'min'}).collect()[0][0]} to {df.agg({'timestamp': 'max'}).collect()[0][0]}")
print(f"  • Fraud transactions: {df.filter(col('is_fraud') == 1).count():,}")
print(f"  • Legitimate transactions: {df.filter(col('is_fraud') == 0).count():,}")

print("\n🎯 Data Architecture:")
print("   Bronze Layer ✓ → workspace.bronze.upi (raw data)")
print("   Silver Layer ⏳ → Next: Feature engineering")
print("   Gold Layer   ⏳ → Next: Model predictions")

print("\n✅ Ready for feature engineering!")
print("   Next: Run 03_feature_engineering notebook")
print("\n" + "="*70)